# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


# Model Building

In [9]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("tourism_project", exist_ok=True)

In [11]:
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

## Data Registration

In [16]:
os.makedirs("tourism_project/data", exist_ok=True)

Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

In [21]:
%%writefile tourism_project/model_building/data_register.py
from huggingface_hub.utils import RepositoryNotFoundError, HfHubHTTPError
from huggingface_hub import HfApi, create_repo
import os


repo_id = "prohra48/tourism-project"
repo_type = "dataset"

# Initialize API client
api = HfApi(token=os.getenv("HF_TOKEN"))

# Step 1: Check if the space exists
try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print(f"Space '{repo_id}' already exists. Using it.")
except RepositoryNotFoundError:
    print(f"Space '{repo_id}' not found. Creating new space...")
    create_repo(repo_id=repo_id, repo_type=repo_type, private=False)
    print(f"Space '{repo_id}' created.")

api.upload_folder(
    folder_path="tourism_project/data",
    repo_id=repo_id,
    repo_type=repo_type,
)

Writing tourism_project/model_building/data_register.py


## Data Preparation

In [24]:
%%writefile tourism_project/model_building/prep.py
# for data manipulation
import pandas as pd
import sklearn
# for creating a folder
import os
# for data preprocessing and pipeline creation
from sklearn.model_selection import train_test_split
# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

# Define constants for the dataset and output paths
api = HfApi(token=os.getenv("HF_TOKEN"))
DATASET_PATH = "hf://datasets/prohra48/tourism-project/tourism.csv"
tourism_dataset = pd.read_csv(DATASET_PATH)
print("Dataset loaded successfully.")

# Define the target variable for the classification task
target = 'ProdTaken'

# Perform data cleaning and remove unnecessary columns
# Dropping CustomerID as it is a unique identifier
if 'CustomerID' in tourism_dataset.columns:
    tourism_dataset = tourism_dataset.drop(columns=['CustomerID'])

# Define predictor matrix (X) using all remaining columns
X = tourism_dataset.drop(columns=[target])

# Define target variable (y)
y = tourism_dataset[target]

# Split dataset into train and test
# Split the dataset into training and test sets
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y,                # Predictors (X) and target variable (y)
    test_size=0.2,       # 20% of the data is reserved for testing
    random_state=42      # Ensures reproducibility by setting a fixed random seed
)

# Save datasets locally
Xtrain.to_csv("Xtrain.csv", index=False)
Xtest.to_csv("Xtest.csv", index=False)
ytrain.to_csv("ytrain.csv", index=False)
ytest.to_csv("ytest.csv", index=False)

files = ["Xtrain.csv", "Xtest.csv", "ytrain.csv", "ytest.csv"]

print("Uploading files to Hugging Face...")
for file_path in files:
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo=file_path.split("/")[-1], # just the filename
        repo_id="prohra48/tourism-project",
        repo_type="dataset",
    )
print("Data Preparation completely successfully!")

Writing tourism_project/model_building/prep.py


## Model Training and Registration with Experimentation Tracking

In [34]:
%%writefile tourism_project/model_building/train.py
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
from huggingface_hub import hf_hub_download, HfApi
import mlflow
import mlflow.sklearn
import joblib
import os

# 1. Setup and Authentication
HF_TOKEN = os.getenv("HF_TOKEN")
DATASET_REPO = "prohra48/tourism-project"
MODEL_REPO = "prohra48/tourism-model" # Make sure to create this model space in Hugging Face!

# 2. Load Train and Test Data from Hugging Face data space
print("Loading data from Hugging Face...")
files = ["Xtrain.csv", "Xtest.csv", "ytrain.csv", "ytest.csv"]
data = {}
for file in files:
    path = hf_hub_download(repo_id=DATASET_REPO, filename=file, repo_type="dataset", token=HF_TOKEN)
    data[file.split('.')[0]] = pd.read_csv(path)

Xtrain, Xtest = data["Xtrain"], data["Xtest"]
# Ensure target variables are 1D arrays/Series
ytrain, ytest = data["ytrain"]["ProdTaken"], data["ytest"]["ProdTaken"] 

# Initialize MLflow experiment
mlflow.set_experiment("Tourism_Package_Prediction")

with mlflow.start_run():
    # 3. Define a model and parameters
     rf = RandomForestClassifier(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],       # Number of trees in the forest
        'max_depth': [5, 10, 20, None],       # How deep the trees can grow
        'min_samples_split': [2, 5, 10],      # Minimum samples required to split a node
        'min_samples_leaf': [1, 2, 4],        # Minimum samples required to form a final leaf (decision)
        'bootstrap': [True, False]            # Whether to use random subsets of data for each tree
    }

    # 4. Tune the model with the defined parameters
    print("Tuning and training the model...")
    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, scoring='accuracy')
    grid_search.fit(Xtrain, ytrain)
    
    # Get the best model
    best_model = grid_search.best_estimator_

    # 5. Evaluate the model performance
    predictions = best_model.predict(Xtest)
    accuracy = accuracy_score(ytest, predictions)
    f1 = f1_score(ytest, predictions)
    
    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")

    # 6. Log all the tuned parameters and metrics
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    # Log the model in MLflow
    mlflow.sklearn.log_model(best_model, "random_forest_model")

# 7. Register the best model in the Hugging Face model hub
print("Saving and uploading best model to Hugging Face...")
joblib.dump(best_model, "model.joblib")

api = HfApi(token=HF_TOKEN)

# Check if model repo exists, create if not
try:
    api.repo_info(repo_id=MODEL_REPO, repo_type="model")
except Exception:
    api.create_repo(repo_id=MODEL_REPO, repo_type="model", private=False)

# Upload the joblib file
api.upload_file(
    path_or_fileobj="model.joblib",
    path_in_repo="model.joblib",
    repo_id=MODEL_REPO,
    repo_type="model"
)

print("Model training, logging, and registration completed successfully!")

Overwriting tourism_project/model_building/train.py


# Deployment

## Dockerfile

In [38]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [40]:
%%writefile tourism_project/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing tourism_project/deployment/Dockerfile


## Streamlit App

Please ensure that the web app script is named `app.py`.

In [53]:
%%writefile tourism_project/deployment/app.py

import streamlit as st
import pandas as pd
import joblib
from huggingface_hub import hf_hub_download

# =========================
# LOAD MODEL FROM HF
# =========================
MODEL_REPO = "prohra48/tourism-model"

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename="model.pkl"
)

model = joblib.load(model_path)

# =========================
# PAGE CONFIG
# =========================
st.set_page_config(page_title="Tourism Predictor", layout="centered")

st.title("🌍 Wellness Tourism Package Predictor")
st.markdown("### Built with MLOps Pipeline 🚀")

st.write("Fill customer details to predict purchase behavior")

# =========================
# SIDEBAR INPUTS
# =========================
st.sidebar.header("Customer Details")

Age = st.sidebar.slider("Age", 18, 70, 30)
TypeofContact = st.sidebar.selectbox("Type of Contact", [0, 1])
CityTier = st.sidebar.selectbox("City Tier", [1, 2, 3])
Occupation = st.sidebar.selectbox("Occupation", [0, 1, 2])
Gender = st.sidebar.selectbox("Gender", [0, 1])
NumberOfPersonVisiting = st.sidebar.slider("Number of Persons Visiting", 1, 10, 2)
PreferredPropertyStar = st.sidebar.selectbox("Preferred Property Star", [1, 2, 3, 4, 5])
MaritalStatus = st.sidebar.selectbox("Marital Status", [0, 1, 2])
NumberOfTrips = st.sidebar.slider("Number of Trips", 0, 10, 2)
Passport = st.sidebar.selectbox("Passport", [0, 1])
OwnCar = st.sidebar.selectbox("Own Car", [0, 1])
NumberOfChildrenVisiting = st.sidebar.slider("Children Visiting", 0, 5, 0)
Designation = st.sidebar.selectbox("Designation", [0, 1, 2, 3, 4])
MonthlyIncome = st.sidebar.number_input("Monthly Income", min_value=1000)
PitchSatisfactionScore = st.sidebar.slider("Pitch Satisfaction Score", 1, 5, 3)
ProductPitched = st.sidebar.selectbox("Product Pitched", [0, 1, 2, 3, 4])
NumberOfFollowups = st.sidebar.slider("Number of Followups", 0, 10, 2)
DurationOfPitch = st.sidebar.slider("Duration of Pitch", 5, 60, 20)

# =========================
# CREATE INPUT DATAFRAME
# =========================
input_data = pd.DataFrame([{
    "Age": Age,
    "TypeofContact": TypeofContact,
    "CityTier": CityTier,
    "Occupation": Occupation,
    "Gender": Gender,
    "NumberOfPersonVisiting": NumberOfPersonVisiting,
    "PreferredPropertyStar": PreferredPropertyStar,
    "MaritalStatus": MaritalStatus,
    "NumberOfTrips": NumberOfTrips,
    "Passport": Passport,
    "OwnCar": OwnCar,
    "NumberOfChildrenVisiting": NumberOfChildrenVisiting,
    "Designation": Designation,
    "MonthlyIncome": MonthlyIncome,
    "PitchSatisfactionScore": PitchSatisfactionScore,
    "ProductPitched": ProductPitched,
    "NumberOfFollowups": NumberOfFollowups,
    "DurationOfPitch": DurationOfPitch
}])

# =========================
# PREDICTION
# =========================
if st.button("Predict"):

    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    st.subheader("Prediction Result")

    if prediction == 1:
        st.success(f"✅ Customer WILL BUY (Confidence: {probability:.2f})")
    else:
        st.error(f"❌ Customer WILL NOT BUY (Confidence: {probability:.2f})")

    st.write("### Input Summary")
    st.dataframe(input_data)

Overwriting tourism_project/deployment/app.py


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

In [55]:
%%writefile tourism_project/deployment/requirements.txt

pandas
numpy
scikit-learn
streamlit
huggingface_hub
datasets
joblib

Overwriting tourism_project/deployment/requirements.txt


# Hosting

In [58]:
os.makedirs("tourism_project/hosting", exist_ok=True)

In [60]:
%%writefile tourism_project/hosting/deploy.py
from huggingface_hub import HfApi
import os

api = HfApi(token=os.getenv("HF_TOKEN"))
SPACE_REPO = "prohra48/tourism-app" # The name of your Hugging Face Space

print("Deploying app to Hugging Face Spaces...")
api.upload_folder(
    folder_path="tourism_project/deployment", # The folder containing your app.py, Dockerfile, and requirements.txt
    repo_id=SPACE_REPO,
    repo_type="space",
)
print("Deployment pushed successfully! Your app is building.")

Writing tourism_project/hosting/deploy.py


# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

```
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server starts
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [ ]:
# Install Git
!apt-get install git

# Set your Git identity (replace with your details)
!git config --global user.email "<-------GitHub Email Address------->"
!git config --global user.name "<--------GitHub UserName--------->"

# Clone your GitHub repository
!git clone https://github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Move your folder to the repository directory
!mv /content/tourism_project/ /content/<--------GitHub Reponame--------->

In [ ]:
# Change directory to the cloned repository
%cd <--------GitHub Reponame--------->/

# Add the new folder to Git
!git add .

# Commit the changes
!git commit -m "first commit"

# Push to GitHub (you'll need your GitHub credentials; use a personal access token if 2FA enabled)
!git push https://<--------GitHub UserName--------->:<--------GitHub Token--------->@github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

<font size=6 color="navyblue">Power Ahead!</font>
___